# Day 6 of ML 30-Days Challenge

# Task

**Intro to ML Pipelines**: Learn to streamline workflows using sklearn.pipeline.Pipeline. Understand how pipelines prevent data leakage from the test set and make your code more reproducible by chaining preprocessing and modeling steps into a single object.

### ML Pipelining Terminology

**ML Pipelining** is the process of automating the workflow of building, training, and deploying machine learning models. It involves chaining together various steps, from data preparation to model deployment, into a single, reproducible process.

**------------------------------------------BETTER LEARN IT FIRST------------------------------------------**

Here are some key terminologies, ordered from basic to more complex:

*   **Step (or Stage):** A single operation within a pipeline, such as data loading, preprocessing, feature extraction, model training, or evaluation.
*   **Workflow:** The overall sequence of operations and dependencies within the pipeline.
*   **Data Preprocessing:** The process of transforming raw data into a format suitable for model training. This can include cleaning, scaling, normalization, and encoding.
*   **Feature Extraction:** The process of selecting or creating features from the raw data that are relevant for the model.
*   **Model Training:** The process of fitting a machine learning model to the prepared data.
*   **Model Evaluation:** The process of assessing the performance of the trained model using metrics.
*   **Hyperparameters:** Parameters of a model or a transformer that are not learned from the data but are set before training. Examples include the number of trees in a random forest or the learning rate of a gradient descent algorithm.
*   **Fit:** The process of training an estimator or transformer on the training data.
*   **Transform:** The process of applying a learned transformation (from the `fit` step) to new data.
*   **Predict:** The process of using a trained model (an estimator) to make predictions on new data.
*   **Estimator:** An object that implements a `fit` method. It learns from data during the `fit` step to build a model. Examples include `LogisticRegression`, `RandomForestClassifier`, etc.
*   **Transformer:** An object that implements `fit` and `transform` methods. It learns from data in the `fit` step and applies a transformation to the data in the `transform` step. Examples include `StandardScaler`, `OneHotEncoder`, etc.
*   **Fit_transform:** A method that combines the `fit` and `transform` steps. It is often used on the training data to learn the transformation and apply it simultaneously.
*   **Pipeline:** A sequence of steps or stages that process data and train a model. Each step's output becomes the next step's input.
*   **Pipeline Object:** The object that represents the entire pipeline, chaining together multiple steps (transformers and estimators).
*   **Steps (in a Pipeline):** The individual components (transformers or estimators) that make up the pipeline. They are typically represented as tuples of a name and the object itself.
*   **Hyperparameter Tuning:** The process of optimizing the hyperparameters of a model to improve its performance.
*   **Cross-validation:** A technique for evaluating the performance of a model by splitting the data into multiple folds and training and testing the model on different combinations of these folds.
*   **Grid Search / Random Search:** Techniques for hyperparameter tuning that involve searching through a predefined space of hyperparameter values to find the best combination.
*   **Model Deployment:** The process of making the trained model available for making predictions on new data.
*   **Artifacts:** The outputs of a pipeline run, such as trained models, evaluation metrics, or processed data.
*   **Model Registry:** A centralized repository for storing and managing trained models.
*   **Deployment:** The process of making a trained model available for inference (making predictions on new data).

## Basic pipeline with preprocessing and model

The Task is to create a simple pipeline, fit it, evaluate it, and print the score. This can be done in a single code block.


In [37]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

# Set a seed as random data from numpy doesn't change
np.random.seed(42)

# Sample data
X = np.random.rand(50, 5)
y = (X.sum(axis=1) > 2.5).astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define a simple pipeline: Scale data, then train Logistic Regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression())
])

# Fit the pipeline
pipeline.fit(X_train, y_train)

# Evaluate
score = pipeline.score(X_test, y_test)
print(f"Pipeline accuracy: {score:.2f}")

Pipeline accuracy: 0.93


## Pipeline with multiple preprocessing steps

 The Task is to create a pipeline with multiple preprocessing steps and a model, then fit and evaluate it.


In [38]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np

# Set a seed as random data from numpy doesn't change
np.random.seed(42)

# Sample data
X = np.random.rand(50, 5)
y = (X.sum(axis=1) > 2.5).astype(int)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Create a pipeline with multiple preprocessing steps: Scale data, `apply PCA` ( <- this new one!! ), then train Logistic Regression
multi_step_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=3)),
    ('logreg', LogisticRegression())
])

# Train the multi-step pipeline
multi_step_pipeline.fit(X_train, y_train)

# Evaluate
multi_step_score = multi_step_pipeline.score(X_test, y_test)
print(f"Multi-step pipeline accuracy: {multi_step_score:.2f}")

Multi-step pipeline accuracy: 0.67


## Pipeline with column transformers

Demonstrate how to apply different transformations to different columns using `ColumnTransformer`.


In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# Set a seed as random data from numpy doesn't change
np.random.seed(42)

# Create a sample DataFrame with mixed data types
data = {'numeric_1': np.random.rand(50),
        'numeric_2': np.random.rand(50),
        'categorical_1': np.random.choice(['A', 'B', 'C'], 50),
        'categorical_2': np.random.choice(['X', 'Y', 'Z'], 50)}

X_mixed = pd.DataFrame(data)
y_mixed = (X_mixed['numeric_1'] + X_mixed['numeric_2'] > 1).astype(int)

# Split the mixed data
X_train_mixed, X_test_mixed, y_train_mixed, y_test_mixed = train_test_split(X_mixed, y_mixed, test_size=0.3, random_state=42)

# Define transformers for numerical and categorical columns
numeric_features = ['numeric_1', 'numeric_2']
numeric_transformer = StandardScaler()

categorical_features = ['categorical_1', 'categorical_2']
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a preprocessor with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Create the full pipeline
col_transformer_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('logreg', LogisticRegression())
])

# Train the pipeline
col_transformer_pipeline.fit(X_train_mixed, y_train_mixed)

# Evaluate the pipeline
col_transformer_score = col_transformer_pipeline.score(X_test_mixed, y_test_mixed)
print(f"ColumnTransformer pipeline accuracy: {col_transformer_score:.2f}")


ColumnTransformer pipeline accuracy: 0.87


## Pipeline with hyperparameter tuning

Implement hyperparameter tuning using GridSearchCV on the existing ColumnTransformer pipeline.


In [49]:
from sklearn.model_selection import GridSearchCV

# Define a parameter grid for GridSearchCV
param_grid = {
    'logreg__C': [0.001, 0.01, 0.1, 1, 10, 100]
}

# Instantiate GridSearchCV
grid_search = GridSearchCV(col_transformer_pipeline, param_grid, cv=5) # cv=5 means 5-fold cross-validation (or something like this)

# Fit GridSearchCV to the training data
grid_search.fit(X_train_mixed, y_train_mixed)

# Print the best hyperparameters
print(f"Best hyperparameters: {grid_search.best_params_}")

# Print the best cross-validation score
print(f"Best cross-validation accuracy: {grid_search.best_score_:.2f}")

# Evaluate the best estimator on the test data
test_score = grid_search.best_estimator_.score(X_test_mixed, y_test_mixed)
print(f"Test set accuracy with best estimator: {test_score:.2f}")

Best hyperparameters: {'logreg__C': 0.1}
Best cross-validation accuracy: 0.89
Test set accuracy with best estimator: 0.87


## Pipeline with cross-validation

Show how to evaluate a pipeline using cross-validation.


In [52]:
from sklearn.model_selection import cross_val_score

# Use cross_val_score to evaluate the pipeline
cv_scores = cross_val_score(col_transformer_pipeline, X_train_mixed, y_train_mixed, cv=5)

# Print the mean and standard deviation of the cross-validation scores
print(f"Cross-validation accuracy: {cv_scores.mean():.2f} +/- {cv_scores.std():.2f}")

Cross-validation accuracy: 0.89 +/- 0.11
